1/ Import data from github shared repository

In [1]:
#Import packages

import pandas as pd
import statsmodels.api as sm
import numpy as np
from tqdm import tqdm
from scipy import stats

In [2]:
#Load the dataset from GitHub

#USe CPI dates from 1958 to Febraury 1972
CPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/CPI_announcement_date.csv"
CPI_announcement_dates = pd.read_csv(CPI_announcement_dates)

#Use PPI dates from 1972 to 2024
PPI_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/PPI_announcement_date.csv"
PPI_announcement_dates = pd.read_csv(PPI_announcement_dates)

#FOMC scheduled interest rate announcement dates from 1990 to 2024
FOMC_announcement_dates = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/FOMC_announcement_date.csv"
FOMC_announcement_dates = pd.read_csv(FOMC_announcement_dates, nrows=106)

#Stock market proxy - CRPS NYSE value-weighted index of all listed shares
NYSE_daily_stock_returns = r"C:/Users/bazel/Documents/Master in Finance/Mémoire/Data/Stocks/NYSE_daily_stock_returns.csv"
NYSE_daily_stock_returns = pd.read_csv(NYSE_daily_stock_returns)

#Stock market proxy - CRPS Amex value-weighted index of all listed shares
Amex_daily_stock_returns = r"C:/Users/bazel/Documents/Master in Finance/Mémoire/Data/Stocks/Amex_daily_stock_returns.csv"
Amex_daily_stock_returns = pd.read_csv(Amex_daily_stock_returns)

#Stock market proxy - CRPS Nasdaq value-weighted index of all listed shares
Nasdaq_daily_stock_returns = r"C:/Users/bazel/Documents/Master in Finance/Mémoire/Data/Stocks/Nasdaq_daily_stock_returns.csv"
Nasdaq_daily_stock_returns = pd.read_csv(Nasdaq_daily_stock_returns)

#Fama_French_25_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Fama_French_25_portfolio_daily ="https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/25_Portfolios_Size_BM_Daily.csv"
Fama_French_25_portfolio_daily_EV = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=18, nrows=25920-19)
Fama_French_25_portfolio_daily_VW = pd.read_csv(Fama_French_25_portfolio_daily, skiprows=25923, nrows=51825-25924)

#Ten_industry_portfolio_daily (Equal_Weighted=EV, Value_Weighted=VW)
Ten_industry_portfolio_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/10_Industry_Portfolios_Daily.csv"
Ten_industry_portfolio_daily_EV = pd.read_csv(Ten_industry_portfolio_daily, skiprows=9, nrows=25911 - 10)
Ten_industry_portfolio_daily_VW = pd.read_csv(Ten_industry_portfolio_daily, skiprows=25914, nrows=51816 - 25915)

#Fama_French_3_factor_daily
Fama_French_3_factor_daily = "https://raw.githubusercontent.com/carolinebazeli/Memoire_HEC/refs/heads/master/F-F_3_Factors_Daily.CSV"
Fama_French_3_factor_daily = pd.read_csv(Fama_French_3_factor_daily, skiprows=4, nrows=25906 - 5)

C:\Users\bazel\AppData\Local\Temp\ipykernel_15392\4052408890.py:17: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  NYSE_daily_stock_returns = pd.read_csv(NYSE_daily_stock_returns)


In [3]:
#Convert NYSE, Amex and Nasdaq stock returns to a Dataframe, where each column is a stock and each row is a date, and the value is the daily return of that stock
# market_data = {
#     'NYSE_daily_stock_returns': NYSE_daily_stock_returns,
#     'Amex_daily_stock_returns': Amex_daily_stock_returns,
#     'Nasdaq_daily_stock_returns': Nasdaq_daily_stock_returns
# }

# for name in market_data:
#     df = market_data[name].drop(columns=['PERMNO', 'EXCHCD'])
#     df['RET'] = pd.to_numeric(df['RET'], errors='coerce')
#     df = df.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
#     df = df.pivot(index='date', columns='TICKER', values='RET').reset_index()
#     market_data[name] = df

# NYSE_daily_stock_returns = market_data['NYSE_daily_stock_returns']
# Amex_daily_stock_returns = market_data['Amex_daily_stock_returns']
# Nasdaq_daily_stock_returns = market_data['Nasdaq_daily_stock_returns']

In [4]:
#Convert NYSE, Amex and Nasdaq stock returns to a Dataframe, where each column is a stock and each row is a date, and the value is the daily return of that stock

NYSE_daily_stock_returns = NYSE_daily_stock_returns.drop(columns=['PERMNO','EXCHCD'])
NYSE_daily_stock_returns['RET'] = pd.to_numeric(NYSE_daily_stock_returns['RET'], errors='coerce')
NYSE_daily_stock_returns = NYSE_daily_stock_returns.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
NYSE_daily_stock_returns = NYSE_daily_stock_returns.pivot(index='date', columns='TICKER', values='RET').reset_index()

Amex_daily_stock_returns = Amex_daily_stock_returns.drop(columns=['PERMNO','EXCHCD'])
Amex_daily_stock_returns['RET'] = pd.to_numeric(Amex_daily_stock_returns['RET'], errors='coerce')
Amex_daily_stock_returns = Amex_daily_stock_returns.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
Amex_daily_stock_returns = Amex_daily_stock_returns.pivot(index='date', columns='TICKER', values='RET').reset_index()

Nasdaq_daily_stock_returns = Nasdaq_daily_stock_returns.drop(columns=['PERMNO','EXCHCD'])
Nasdaq_daily_stock_returns['RET'] = pd.to_numeric(Nasdaq_daily_stock_returns['RET'], errors='coerce')
Nasdaq_daily_stock_returns = Nasdaq_daily_stock_returns.groupby(['date', 'TICKER'])['RET'].mean().reset_index()
Nasdaq_daily_stock_returns = Nasdaq_daily_stock_returns.pivot(index='date', columns='TICKER', values='RET').reset_index()


In [5]:
#Convert date columns to datetime format
CPI_announcement_dates = CPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
CPI_announcement_dates['A_date'] = pd.to_datetime(CPI_announcement_dates['A_date']).dt.date

PPI_announcement_dates = PPI_announcement_dates.rename(columns = {'Release Dates': 'A_date'})
PPI_announcement_dates['A_date'] = pd.to_datetime(PPI_announcement_dates['A_date']).dt.date

FOMC_announcement_dates = FOMC_announcement_dates.rename(columns = {'FOMC Meeting Date': 'A_date'}).reset_index(drop=True)
FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date

NYSE_daily_stock_returns = NYSE_daily_stock_returns.rename(columns = {'date': 'Date'})
NYSE_daily_stock_returns['Date'] = pd.to_datetime(NYSE_daily_stock_returns['Date'], errors='coerce').dt.date

Amex_daily_stock_returns = Amex_daily_stock_returns.rename(columns = {'date': 'Date'})
Amex_daily_stock_returns['Date'] = pd.to_datetime(Amex_daily_stock_returns['Date'], errors='coerce').dt.date

Nasdaq_daily_stock_returns = Nasdaq_daily_stock_returns.rename(columns = {'date': 'Date'})
Nasdaq_daily_stock_returns['Date'] = pd.to_datetime(Nasdaq_daily_stock_returns['Date'], errors='coerce').dt.date

Fama_French_25_portfolio_daily_EV = Fama_French_25_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_EV['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_25_portfolio_daily_VW = Fama_French_25_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_25_portfolio_daily_VW['Date'] = pd.to_datetime(Fama_French_25_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_EV = Ten_industry_portfolio_daily_EV.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_EV['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_EV['Date'].astype(str), format='%Y%m%d').dt.date

Ten_industry_portfolio_daily_VW = Ten_industry_portfolio_daily_VW.rename(columns = {'Unnamed: 0': 'Date'})
Ten_industry_portfolio_daily_VW['Date'] = pd.to_datetime(Ten_industry_portfolio_daily_VW['Date'].astype(str), format='%Y%m%d').dt.date

Fama_French_3_factor_daily = Fama_French_3_factor_daily.rename(columns = {'Unnamed: 0': 'Date'})
Fama_French_3_factor_daily['Date'] = pd.to_datetime(Fama_French_3_factor_daily['Date'].astype(str), format='%Y%m%d').dt.date

C:\Users\bazel\AppData\Local\Temp\ipykernel_15392\2790959862.py:9: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  FOMC_announcement_dates['A_date'] = pd.to_datetime(FOMC_announcement_dates['A_date']).dt.date


In [6]:
#Merge the three dataset NYSE, Amex and Nasdaq into a single one:

stock_market_daily_returns = NYSE_daily_stock_returns.merge(
    Amex_daily_stock_returns, on='Date', how='outer'
).merge(
    Nasdaq_daily_stock_returns, on='Date', how='outer'
)

cols = ['Date'] + sorted([col for col in stock_market_daily_returns.columns if col != 'Date'])
stock_market_daily_returns = stock_market_daily_returns[cols]

2/ Split data between announcement days and non-announcement days for each portfolio

In [7]:
#Select CPI_announcement_dates up until February 1972
CPI_announcement_dates = CPI_announcement_dates.loc[CPI_announcement_dates['A_date'] <= pd.to_datetime('1972-02-01').date()]
CPI_announcement_dates = CPI_announcement_dates.reset_index(drop=True)

#Select PPI_announcement_dates from February 1972
PPI_announcement_dates = PPI_announcement_dates.loc[PPI_announcement_dates['A_date'] > pd.to_datetime('1972-02-01').date()]
PPI_announcement_dates = PPI_announcement_dates.reset_index(drop=True)

In [8]:
all_A_dates = pd.concat([
    CPI_announcement_dates['A_date'],
    FOMC_announcement_dates['A_date'],
    PPI_announcement_dates['A_date']
])

stock_market_daily_returns_A_Day = stock_market_daily_returns[stock_market_daily_returns['Date'].isin(all_A_dates)].reset_index(drop=True)
stock_market_daily_returns_N_Day = stock_market_daily_returns[~stock_market_daily_returns['Date'].isin(all_A_dates)].reset_index(drop=True)

Fama_French_25_portfolio_daily_EV_A_Day = Fama_French_25_portfolio_daily_EV[Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_A_Day = Fama_French_25_portfolio_daily_VW[Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_A_Day = Ten_industry_portfolio_daily_VW[Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_A_Day = Ten_industry_portfolio_daily_EV[Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)

Fama_French_25_portfolio_daily_EV_N_Day = Fama_French_25_portfolio_daily_EV[~Fama_French_25_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)
Fama_French_25_portfolio_daily_VW_N_Day = Fama_French_25_portfolio_daily_VW[~Fama_French_25_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_VW_N_Day = Ten_industry_portfolio_daily_VW[~Ten_industry_portfolio_daily_VW['Date'].isin(all_A_dates)].reset_index(drop=True)
Ten_industry_portfolio_daily_EV_N_Day = Ten_industry_portfolio_daily_EV[~Ten_industry_portfolio_daily_EV['Date'].isin(all_A_dates)].reset_index(drop=True)


In [9]:
#add the daily market Equity Risk premium and risk-free rate to all the portfolios

Fama_French_3_factor_daily = Fama_French_3_factor_daily.rename(columns={'Mkt-RF': 'Equity Risk Premium', 'RF': 'Risk-Free Rate'})

df_names = [
    'stock_market_daily_returns',
    'stock_market_daily_returns_A_Day',
    'stock_market_daily_returns_N_Day',
    'Fama_French_25_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_VW',
    'Ten_industry_portfolio_daily_EV',
    'Fama_French_25_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_VW_A_Day',
    'Ten_industry_portfolio_daily_EV_A_Day',
    'Fama_French_25_portfolio_daily_EV_N_Day',
    'Fama_French_25_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_VW_N_Day',
    'Ten_industry_portfolio_daily_EV_N_Day'
]

#Convert to decimal format (from percent) to match stock's returns format 
Fama_French_3_factor_daily[['Equity Risk Premium', 'Risk-Free Rate']] /= 100

ff3_cols = ['Date', 'Equity Risk Premium', 'Risk-Free Rate']

for name in df_names:
    globals()[name] = globals()[name].merge(Fama_French_3_factor_daily[ff3_cols], on='Date', how='left')

In [10]:
#Select stock market returns only from 2010 to 2024 - to test
stock_market_daily_returns['Date'] = pd.to_datetime(stock_market_daily_returns['Date'])
stock_market_daily_returns_2010 = stock_market_daily_returns.loc[stock_market_daily_returns['Date'] >= pd.to_datetime('2010-01-01')].reset_index(drop=True)

stock_market_daily_returns_A_Day['Date'] = pd.to_datetime(stock_market_daily_returns_A_Day['Date'])
stock_market_daily_returns_2010_A_Day = stock_market_daily_returns_A_Day.loc[stock_market_daily_returns_A_Day['Date'] >= pd.to_datetime('2010-01-01')].reset_index(drop=True)

stock_market_daily_returns_N_Day['Date'] = pd.to_datetime(stock_market_daily_returns_N_Day['Date'])
stock_market_daily_returns_2010_N_Day = stock_market_daily_returns_N_Day.loc[stock_market_daily_returns_N_Day['Date'] >= pd.to_datetime('2010-01-01')].reset_index(drop=True)

In [11]:
#Select stock market returns only from 1990 to 2024 - to test
stock_market_daily_returns['Date'] = pd.to_datetime(stock_market_daily_returns['Date'])
stock_market_daily_returns_1990 = stock_market_daily_returns.loc[stock_market_daily_returns['Date'] >= pd.to_datetime('1990-01-01')].reset_index(drop=True)

stock_market_daily_returns_A_Day['Date'] = pd.to_datetime(stock_market_daily_returns_A_Day['Date'])
stock_market_daily_returns_1990_A_Day = stock_market_daily_returns_A_Day.loc[stock_market_daily_returns_A_Day['Date'] >= pd.to_datetime('1990-01-01')].reset_index(drop=True)

stock_market_daily_returns_N_Day['Date'] = pd.to_datetime(stock_market_daily_returns_N_Day['Date'])
stock_market_daily_returns_1990_N_Day = stock_market_daily_returns_N_Day.loc[stock_market_daily_returns_N_Day['Date'] >= pd.to_datetime('1990-01-01')].reset_index(drop=True)


1/ Methods for computing stock market betas

In [12]:
#Step 1a - Estimate unconditionnal full-sample beta through a time-series regression - run CAPM regression on all daily data for each stock

def compute_rolling_betas_from_dataset(data, window=252):
    """
    Compute 1-year rolling betas using a dataset where:
    - rows are dates,
    - columns: [stock1, ..., stockN, 'Equity Risk Premium', 'Risk-Free Rate']

    Returns:
    - DataFrame with MultiIndex (date, ticker) and column ['beta']
    """
    required_columns = ['Equity Risk Premium', 'Risk-Free Rate']
    stock_columns = data.columns.difference(required_columns)
    
    betas = []
    end_of_months = data.resample('M').last().index

    for current_date in end_of_months:
        try:
            start_idx = data.index.get_loc(current_date)
        except KeyError:
            continue

        if start_idx < window:
            continue
        
        window_data = data.iloc[start_idx - window:start_idx]

        if len(window_data) < 200:
            continue

        X = sm.add_constant(window_data['Equity Risk Premium'])

        for stock in stock_columns:
            y = window_data[stock] - window_data['Risk-Free Rate']

            if y.isnull().sum() > 50:
                continue

            try:
                model = sm.OLS(y, X, missing='drop')
                results = model.fit()
                beta = results.params['Equity Risk Premium']
                betas.append((current_date, stock, beta))
            except:
                continue

    betas_time_varying = pd.DataFrame(betas, columns=['date', 'ticker', 'beta'])
    return betas_time_varying.pivot(index='date', columns='ticker', values='beta').sort_index()


In [13]:
#Step 1b - Estimate One-Year rolling window Betas (Time-Varying betas) on all daily data for each stock (unconditionnal betas)

def rolling_betas(data, window=252):
    """
    Compute 1-year rolling betas using a dataset where:
    - rows are dates (index),
    - columns: [stock1, ..., stockN, 'Equity Risk Premium', 'Risk-Free Rate']

    Returns:
    - DataFrame with MultiIndex (date, ticker) and column ['beta']
    """

    if 'Date' in data.columns:
        data.set_index('Date', inplace=True)
        data.index = pd.to_datetime(data.index, errors='coerce')
        
    required_columns = ['Equity Risk Premium', 'Risk-Free Rate']
    stock_columns = data.columns.difference(required_columns)
    
    betas = []
    
    # Resample to get the end of each month
            #end_of_months = data.resample('M').last().index
            #end_of_months = pd.date_range(start=data.index.min(), end=data.index.max(), freq='BM')
            #end_of_months = [d for d in end_of_months if d in data.index]

    end_of_months = data.groupby([data.index.year, data.index.month]).apply(lambda x: x.index.max()).sort_values()
    
    for current_date in tqdm(end_of_months, desc='Rolling beta estimation'):
        
        try:
            start_idx = data.index.get_loc(current_date)
        except KeyError:
            continue

        if start_idx < window:
            continue
        
        window_data = data.iloc[start_idx - window:start_idx]

        if len(window_data) < 200:
            continue

        X = sm.add_constant(window_data['Equity Risk Premium'])

        for stock in stock_columns:
            y = window_data[stock] - window_data['Risk-Free Rate']

            if y.isnull().sum() > 50:
                continue

            try:
                model = sm.OLS(y, X, missing='drop')
                results = model.fit()
                beta = results.params['Equity Risk Premium']
                betas.append((current_date, stock, beta))
            except:
                continue
              
    betas_time_varying = pd.DataFrame(betas, columns=['date', 'ticker', 'beta'])
    return betas_time_varying.set_index(['date', 'ticker'])['beta'].unstack()


2/ Beta in Cross-Sectional Asset Pricing Regressions

In [14]:
#Step 2a - Fama-MacBeth Two-Step Regressions (by Announcement vs Non-Announcement Days)


def fama_macbeth_cross_sectional(data_subset, rolling_betas_df):
    """
    Fama–MacBeth cross-sectional regression with 'Date' as column.

    Parameters:
        data_subset: pd.DataFrame (like Fama_French_25_portfolio_daily_EV_A_Day) - the data subset for the regression, split between A_days and N_days
            Must include 'Date', 25 portfolios, 'Equity Risk Premium', and 'Risk-Free Rate'.
        rolling_betas_df: pd.DataFrame (like ama_French_25_portfolio_daily_EV_rolling_betas) - contains the rolling beta for the whole period
            Output of rolling_betas(): index = date (datetime), columns = gamma_0, gamma_1.

    Returns:
        pd.DataFrame:
            Index = dates (t+1), columns = ['gamma_0', 'gamma_1']
    """
    # Ensure datetime and sorted
    if 'Date' in data_subset.columns:
        data_subset['Date'] = pd.to_datetime(data_subset['Date'])
        data_subset = data_subset.sort_values('Date').reset_index(drop=True)
    else:
        data_subset = data_subset.reset_index()  # assumes index is Date
        data_subset.rename(columns={data_subset.columns[0]: 'Date'}, inplace=True)
        data_subset['Date'] = pd.to_datetime(data_subset['Date'])
        data_subset = data_subset.sort_values('Date').reset_index(drop=True)

    results = []

    required_cols = ['Equity Risk Premium', 'Risk-Free Rate', 'Date']
    portfolio_cols = data_subset.columns.difference(required_cols)

    for t in tqdm(range(1, len(data_subset)), desc="Fama-MacBeth regression"):
        date_t = data_subset.loc[t, 'Date']

        # Find latest beta before date_t
        beta_dates = rolling_betas_df.index[rolling_betas_df.index <= date_t]
        if beta_dates.empty:
            continue
        date_beta = beta_dates.max()
        betas_t = rolling_betas_df.loc[date_beta]

        # Excess returns at t
        excess_returns = data_subset.loc[t, portfolio_cols] - data_subset.loc[t, 'Risk-Free Rate']

        # Match assets
        common_assets = excess_returns.dropna().index.intersection(betas_t.dropna().index)
        if len(common_assets) < 5:
            continue

        y = excess_returns[common_assets].astype(float)
        X = betas_t[common_assets].astype(float)
        X = sm.add_constant(X)

        # Drop any remaining NaNs
        valid_idx = y.index.intersection(X.dropna().index)
        y = y.loc[valid_idx]
        X = X.loc[valid_idx]

        model = sm.OLS(y, X).fit()

        results.append({
            'Date': date_t,
            'gamma_0': model.params['const'],
            'gamma_1': model.params[X.columns[1]],
            'R_squared': model.rsquared
        })

    return pd.DataFrame(results).set_index('Date')


In [15]:
#Step 2a bis - Fama-MacBeth Two-Step Regressions (by Announcement vs Non-Announcement Days) - Table Output of findings

def summarize_fama_macbeth_results(gammas_A, gammas_N):
    """
    Produce summary stats like academic table for Fama-MacBeth regressions.

    Inputs:
        gammas_A: DataFrame with columns ['gamma_0', 'gamma_1'], index = dates (A-days)
        gammas_N: DataFrame with columns ['gamma_0', 'gamma_1'], index = dates (N-days)

    Output:
        summary_df: Formatted table with mean coefficients, t-stats, R-squared and differences
    """


    def mean_se_t(series):
        mean = series.mean()
        se = series.std(ddof=1) / np.sqrt(len(series))
        t_stat = mean / se
        return mean, t_stat

    # Intercept
    gamma0_A, t0_A = mean_se_t(gammas_A['gamma_0'])
    gamma0_N, t0_N = mean_se_t(gammas_N['gamma_0'])
    diff_gamma0 = gamma0_A - gamma0_N
    t_diff_0, _ = stats.ttest_ind(gammas_A['gamma_0'], gammas_N['gamma_0'], equal_var=False)

    # Slope (beta)
    gamma1_A, t1_A = mean_se_t(gammas_A['gamma_1'])
    gamma1_N, t1_N = mean_se_t(gammas_N['gamma_1'])
    diff_gamma1 = gamma1_A - gamma1_N
    t_diff_1, _ = stats.ttest_ind(gammas_A['gamma_1'], gammas_N['gamma_1'], equal_var=False)

    # Average R²
    r2_A = gammas_A['R_squared'].mean()
    r2_N = gammas_N['R_squared'].mean()
    r2_diff = r2_A - r2_N

    # Build final DataFrame
    data = {
        "Intercept": [gamma0_A, gamma0_N, diff_gamma0],
        "Intercept t-stat": [t0_A, t0_N, t_diff_0],
        "Beta": [gamma1_A, gamma1_N, diff_gamma1],
        "Beta t-stat": [t1_A, t1_N, t_diff_1],
        "Avg R²": [r2_A, r2_N, r2_diff]
    }

    index = ['a-day', 'n-day', 'a-day − n-day']
    summary_df = pd.DataFrame(data, index=index)

    return summary_df

In [29]:
results_stock_market_daily_returns_1990 = summarize_fama_macbeth_results(gammas_A, gammas_N)
results_stock_market_daily_returns_1990

,Intercept,Intercept t-stat,Beta,Beta t-stat,Avg R²
a-day,-0.009997,-22.201697,0.001258,0.025929,0.019490
n-day,-0.008806,-81.448420,-0.026019,-2.279028,0.017946
a-day − n-day,-0.001191,-2.572371,0.027277,0.547099,0.001544


In [21]:
Ten_industry_portfolio_daily_VW_rolling_betas = rolling_betas(Ten_industry_portfolio_daily_VW)
gammas_A = fama_macbeth_cross_sectional(Ten_industry_portfolio_daily_VW_A_Day, Ten_industry_portfolio_daily_VW_rolling_betas)
gammas_N = fama_macbeth_cross_sectional(Ten_industry_portfolio_daily_VW_N_Day, Ten_industry_portfolio_daily_VW_rolling_betas)
results_Ten_industry_portfolio_daily_VW = summarize_fama_macbeth_results(gammas_A, gammas_N)
results_Ten_industry_portfolio_daily_VW

Fama-MacBeth regression: 100%|██████████| 24957/24957 [00:51<00:00, 481.37it/s]


,Intercept,Intercept t-stat,Beta,Beta t-stat,Avg R²
a-day,0.045381,1.233066,0.000509,0.933824,0.273191
n-day,0.070674,8.932415,0.000040,0.354566,0.249103
a-day − n-day,-0.025293,-0.671889,0.000469,0.843581,0.024087


In [ ]:
stock_market_daily_returns_rolling_betas = rolling_betas(stock_market_daily_returns)
gammas_A = fama_macbeth_cross_sectional(stock_market_daily_returns_A_Day, stock_market_daily_returns_rolling_betas)
gammas_N = fama_macbeth_cross_sectional(stock_market_daily_returns_N_Day, stock_market_daily_returns_rolling_betas)
results_stock_market_daily_returns= summarize_fama_macbeth_results(gammas_A, gammas_N)
results_stock_market_daily_returns

In [15]:
stock_market_daily_returns_rolling_betas = rolling_betas(stock_market_daily_returns)
gammas_A = fama_macbeth_cross_sectional(stock_market_daily_returns_A_Day, stock_market_daily_returns_rolling_betas)
gammas_N = fama_macbeth_cross_sectional(stock_market_daily_returns_N_Day, stock_market_daily_returns_rolling_betas)
results_stock_market_daily_returns= summarize_fama_macbeth_results(gammas_A, gammas_N)
results_stock_market_daily_returns

Fama-MacBeth regression: 100%|██████████| 18318/18318 [07:41<00:00, 39.73it/s]


,Intercept,Intercept t-stat,Beta,Beta t-stat,Avg R²
a-day,-0.015401,-32.584862,0.005005,0.144050,0.016595
n-day,-0.016438,-148.540653,-0.019229,-2.679531,0.015150
a-day − n-day,0.001037,2.135655,0.024234,0.683025,0.001445


Method 2: Compute beta-sorted portfolio

In [17]:
#Step 2b - Function to build equally-weighted beta-sorted portfolios

def build_beta_decile_portfolios(rolling_betas, stock_returns):
    """
    Create 10 equally weighted beta-sorted portfolios based on monthly rolling betas.

    Parameters:
    - rolling_betas: DataFrame (index = end-of-month dates, columns = stocks, values = betas)
    - stock_returns: DataFrame (index = daily dates, columns = same stocks, values = daily returns)

    Returns:
    - DataFrame: index = daily dates, columns = 'Decile 1', ..., 'Decile 10', values = portfolio returns
    """

    portfolios = []

    # Make sure indexes are datetime and sorted
    rolling_betas = rolling_betas.sort_index()
    stock_returns = stock_returns.sort_index()
    
    for date in rolling_betas.index:
        # Get next month’s date range
        start = date + pd.Timedelta(days=1)
        try:
            end = rolling_betas.index[rolling_betas.index.get_loc(date) + 1]
        except IndexError:
            break  # no next month

        # Get beta values for this rebalancing date
        betas = rolling_betas.loc[date].dropna()

        # Assign each stock to a decile based on beta
        deciles = pd.qcut(betas, 10, labels=[f'Decile {i+1}' for i in range(10)])

        # Get returns for the next month
        monthly_returns = stock_returns.loc[start:end]
        if monthly_returns.empty:
            continue

        # Build portfolios
        decile_returns = {}
        for decile in deciles.unique():
            stocks_in_decile = deciles[deciles == decile].index
            # Select return data for these stocks
            decile_data = monthly_returns[stocks_in_decile].copy()
            # Calculate average daily return across the decile (equally weighted)
            decile_returns[decile] = decile_data.mean(axis=1)

        portfolios.append(pd.DataFrame(decile_returns))

    # Combine all monthly DataFrames
    all_portfolios = pd.concat(portfolios).sort_index()

    # Add Equiry Risk Premium and Risk-Free Rate from Fama-French 3-factor model
    Fama_French_3_factor_daily['Date'] = pd.to_datetime(Fama_French_3_factor_daily['Date'], errors='coerce')
    ff3_cols = ['Date', 'Equity Risk Premium', 'Risk-Free Rate']
    all_portfolios = all_portfolios.merge(Fama_French_3_factor_daily[ff3_cols], left_index=True, right_on='Date', how='left')
    
    return all_portfolios

In [18]:
#Test beta-sorted portfolio

#Compute rolling betas for the stock market daily returns
stock_market_daily_returns_2010_rolling_betas = rolling_betas(stock_market_daily_returns_2010)

Rolling beta estimation: 100%|██████████| 180/180 [15:30<00:00,  5.17s/it]


NameError: name 'stock_market_daily_returns_1990_rolling_betas' is not defined

In [ ]:
#Build beta-sorted portfolios from 2010
Beta_sorted_portfolios = build_beta_decile_portfolios(stock_market_daily_returns_2010_rolling_betas,stock_market_daily_returns_2010)

#Compute rolling betas for the portfolios
Beta_sorted_portfolios_rolling_betas = rolling_betas(Beta_sorted_portfolios)

#Split the portfolios into A and N days
Beta_sorted_portfolios_A_Day = Beta_sorted_portfolios[Beta_sorted_portfolios.index.isin(stock_market_daily_returns_A_Day['Date'])]
Beta_sorted_portfolios_N_Day = Beta_sorted_portfolios[Beta_sorted_portfolios.index.isin(stock_market_daily_returns_N_Day['Date'])]


#Compute Fama-MacBeth regression for the portfolios
gammas_A = fama_macbeth_cross_sectional(Beta_sorted_portfolios_A_Day, Beta_sorted_portfolios_rolling_betas)
gammas_N = fama_macbeth_cross_sectional(Beta_sorted_portfolios_N_Day, Beta_sorted_portfolios_rolling_betas)

#Summarize results
results_Beta_sorted_portfolios = summarize_fama_macbeth_results(gammas_A, gammas_N)
results_Beta_sorted_portfolios

Fama-MacBeth regression: 100%|██████████| 3300/3300 [00:06<00:00, 502.63it/s]


,Intercept,Intercept t-stat,Beta,Beta t-stat,Avg R²
a-day,-0.000569,-0.846673,-0.002199,-1.884775,0.653076
n-day,0.000632,5.176490,-0.000023,-0.092203,0.612510
a-day − n-day,-0.001201,-1.758572,-0.002176,-1.822554,0.040566
